# This is an enhanced the career conversation chatbox using Gradio


This is solution has the following features

| Sno | Features |
|-|---|
| 1 | Resume-aware chat agent|
| 2 | Tool use (push notifications) |
| 3 |  **SQLite logging** (emails + unknown questions)  |
| 4 |  **LLM Evaluator** (third-party judge with gemini-2.0-flash-001)|


---

### What gets logged to SQLite:
- **Interested users** — name, email, notes, timestamp
- **Unknown questions** — questions the agent couldn't answer, timestamp

> 💡 You can use the captured unknown questions later to expand your RAG knowledge base!

### Setup checklist:
1. Download your LinkedIn profile as a PDF and upload it to Google Drive, then add the file ID to your `.env` as `LINKEDIN_FILE_ID`
2. Write a short bio about yourself, save it as a `.txt` file, upload it to Google Drive, then add the file ID to your `.env` as `SUMMARY_FILE_ID`
3. Change `YOUR_NAME` in the notebook to your actual name
4. Add Pushover keys to your `.env` file (`PUSHOVER_USER` and `PUSHOVER_TOKEN`)
5. Add `OPENROUTER_API_KEY` to your `.env` for the evaluator (or swap the model for any other LLM available on OpenRouter)

> **Note:** Upload both files to Google Drive, share each as **Anyone with the link (Viewer)**, then copy the file ID from the share URL — it is the long string between `/d/` and `/view`: `https://drive.google.com/file/d/`**FILE_ID**`/view`

## 1. Imports & Initialization

In [45]:
# Core imports
import os
import json
import sqlite3
import requests
import gdown
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from pydantic import BaseModel
import gradio as gr

load_dotenv(override=True)

# Primary LLM (GPT) 

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()


# Evaluator LLM (OpenRouter) 
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set")

evaluator_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)
EVALUATOR_MODEL = "google/gemini-2.0-flash-001"  # or swap for any OpenRouter model

# Pushover
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

OpenAI API Key exists and begins sk-proj-
OpenRouter API Key exists and begins sk-or-v1
Pushover user found and starts with u
Pushover token found and starts with a


## 2. SQLite Database Setup

Two tables created and initialized:
- `interested_users` — captures leads (email, name, conversation notes)
- `unknown_questions` — captures gaps in your resume/knowledge base

In [2]:
DB = "career_chatbot.db"

def init_db():
    """Create the database tables if they don't already exist."""
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS interested_users (
                id        INTEGER PRIMARY KEY AUTOINCREMENT,
                email     TEXT NOT NULL,
                name      TEXT DEFAULT 'Not provided',
                notes     TEXT DEFAULT 'None',
                timestamp TEXT NOT NULL
            )
        ''')

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS unknown_questions (
                id        INTEGER PRIMARY KEY AUTOINCREMENT,
                question  TEXT NOT NULL,
                timestamp TEXT NOT NULL
            )
        ''')

        conn.commit()
    print(f"✅ Database '{DB}' initialised.")

init_db()

✅ Database 'career_chatbot.db' initialised.


## 3. Pushover Function


In [ ]:
def push(message):
    print(f"Push: {message}")
    if not (pushover_user and pushover_token):
        print("(Pushover not configured - skipping)")
        return
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    try:
        requests.post(pushover_url, data=payload, timeout=5)
    except Exception as e:
        print(f"Push failed: {e}")

## 4. Tool Functions

Each function does **three things**: sends a push notification, logs to SQLite, and returns a confirmation to the LLM.

In [28]:

def record_user_details(email, name="Name not provided", notes="not provided"):
    """
    Called by the LLM when a user shares their email address.
    1. Sends a push notification
    2. Persists the lead to SQLite
    """
    timestamp = datetime.now().isoformat()
    push(f"Recording interest from {name} with email {email} \n Notes: {notes} at {timestamp}")
    with sqlite3.connect(DB) as conn:
        conn.execute(
            "INSERT INTO interested_users (email, name, notes, timestamp) VALUES (?, ?, ?, ?)",
            (email, name, notes, timestamp)
        )
        conn.commit()
    print(f" Saved user: {name} <{email}>")
    return {"recorded": "ok", "message": "User details saved successfully"}




def record_unknown_question(question):
    timestamp = datetime.now().isoformat()
    push(f"Recording unknown question: {question} at {timestamp}")
    with sqlite3.connect(DB) as conn:
        conn.execute(
            "INSERT INTO unknown_questions (question, timestamp) VALUES (?, ?)",
            (question, timestamp)
        )
        conn.commit()
    print(f" Saved unknown question: {question}")
    return {"recorded": "ok", "message": "Question logged for future improvement"}

## 5. Tool Definitions (to be used by OPENAI LLM)

In [5]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address. "
                   "Always try to collect their name and any relevant context about why they reached out.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            },
            "notes": {
                "type": "string",
                "description": "Any additional context: are they a recruiter, potential client, collaborator? What did they ask about?"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered because the information "
                   "wasn't available in the provided resume or summary context. This helps improve the knowledge base.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The exact question that couldn't be answered"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

print("✅ Tool definitions ready.")

✅ Tool definitions ready.


## 6. Tool Call Handler

In [6]:
def handle_tool_calls(tool_calls):
    """Execute tool calls returned by the LLM and return results in OpenAI format."""
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"🔧 Tool called: {tool_name} with args: {arguments}", flush=True)

        # Dispatch to the correct Python function
        tool_fn = globals().get(tool_name)
        result = tool_fn(**arguments) if tool_fn else {"error": f"Unknown tool: {tool_name}"}

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

## 7. Load Resume Context

Update `YOUR_NAME` to your actual name. The notebook will automatically download your LinkedIn PDF and summary from Google Drive using the file IDs stored in your `.env` file (`LINKEDIN_FILE_ID` and `SUMMARY_FILE_ID`).

In [ ]:
# ── CHANGE THIS ──────────────────────────────────────────────
YOUR_NAME = "Gerald Okeke"   # Replace with your name
# ─────────────────────────────────────────────────────────────

LINKEDIN_FILE_ID = os.getenv("LINKEDIN_FILE_ID") #Load from .env your linkedin file id uploaded to google drive
SUMMARY_FILE_ID  = os.getenv("SUMMARY_FILE_ID") #Load from .env your summary file id uploaded to google drive

gdown.download(f"https://drive.google.com/uc?id={LINKEDIN_FILE_ID}", "linkedin.pdf", quiet=False)
gdown.download(f"https://drive.google.com/uc?id={SUMMARY_FILE_ID}",  "summary.txt",  quiet=False)

# Load LinkedIn PDF

try:
    reader = PdfReader("linkedin.pdf")
    linkedin = "".join(
        page.extract_text() for page in reader.pages if page.extract_text()
    )
    print(f"✅ LinkedIn PDF loaded ({len(linkedin)} characters)")
except FileNotFoundError:
    linkedin = "LinkedIn profile not provided."
    print("⚠️  linkedin.pdf not found — using placeholder")

# Load summary
try:
    with open("summary.txt", "r", encoding="utf-8") as f:
        summary = f.read()
    print(f"✅ Summary loaded ({len(summary)} characters)")
except FileNotFoundError:
    summary = "Summary not provided."
    print("⚠️  summary.txt not found — using placeholder")




## 8. System Prompts

In [48]:
# Agent system prompt

system_prompt = f"You are acting as {YOUR_NAME}. You are answering questions on {YOUR_NAME}'s website, \
particularly questions related to {YOUR_NAME}'s career, background, skills and experience. \
Your responsibility is to represent {YOUR_NAME} for interactions on the website as faithfully as possible. \
You are given a summary of {YOUR_NAME}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their name and email and record it using your record_user_details tool. \
If they only provide an email without a name, that is fine - record it anyway."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {YOUR_NAME}."



# Evaluator system prompt

evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {YOUR_NAME} and is representing {YOUR_NAME} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {YOUR_NAME} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += "With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

## 9. Evaluator (Gemini as the Judge)

Uses structured output (Pydantic) to get a machine-readable verdict on every response.

In [49]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

def evaluate(reply, message, history):
    if not openrouter_api_key:
        print("Evaluator skipped (no OPENROUTER_API_KEY)")
        return None
    try:
        messages = [
            {"role": "system", "content": evaluator_system_prompt
                + "\n\nRespond ONLY with a valid JSON object with keys: "
                  "is_acceptable (bool), feedback (str). "
                  "No markdown, no backticks, just raw JSON."},
            {"role": "user", "content": evaluator_user_prompt(reply, message, history)}
        ]
        response = evaluator_client.chat.completions.create(
            model=EVALUATOR_MODEL,
            messages=messages,
            max_tokens=300
        )
        raw = response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
            raw = raw.strip()
        data = json.loads(raw)
        return Evaluation(**data)
    except Exception as e:
        print(f"Evaluation error: {e}")
        return None

## 11. Main Chat Function



In [50]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    done = False
    while not done:
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message_obj = response.choices[0].message
            tool_results = handle_tool_calls(message_obj.tool_calls)
            messages.append(message_obj)
            messages.extend(tool_results)
        else:
            done = True

    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    if evaluation is None:
        pass
    elif evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - returning reply")
        print(evaluation.feedback)

    return reply

## 12. Launch the Gradio Interface

In [ ]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title=f"Chat with {YOUR_NAME}",
    description=f"Ask me anything about my career, skills, and experience. I'm {YOUR_NAME}!",
    examples=[
        "What's your professional background?",
        "What technologies do you specialize in?",
        "Are you open to new opportunities?",
        "How can I get in touch with you?"
    ]
).launch()

## 13. Inspect Your Database

Run these cells at any time to review what's been captured in your database.

In [ ]:
# View all captured emails of clients
print("=" * 60)
print("Emails of potential clients")
print("=" * 60)
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT id, name, email, notes, timestamp FROM interested_users ORDER BY timestamp DESC")
    rows = cursor.fetchall()

if rows:
    for row in rows:
        print(f"[{row[0]}] {row[1]} | {row[2]}")
        print(f"     Notes: {row[3]}")
        print(f"     Time:  {row[4]}")
        print()
else:
    print("No emails captured yet.")

In [ ]:
# View all unknown questions (your RAG improvement backlog!)
print("UNKNOWN QUESTIONS (To help RAG Improvement )")
print("=" * 60)
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT id, question, timestamp FROM unknown_questions ORDER BY timestamp DESC")
    rows = cursor.fetchall()

if rows:
    for row in rows:
        print(f"[{row[0]}] {row[2]}")
        print(f"     Q: {row[1]}")
        print()
else:
    print("No unknown questions captured yet.")